In [1]:
%cd ../..

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


In [2]:
from postprocess.ia.eval import (
    answers_for_model,
    fm_scores_for_model,
    metrics_for_model,
    metrics_for_models
)

In [3]:
MODEL = "qwen2_5_vl_7b"
task1 = ("strokerehab_ia1_3_30", "strokerehab_ia1_31_33")  # simultaneous
task2 = ("strokerehab_ia2_3_30", "strokerehab_ia2_31_33")  # individual

print(metrics_for_model("qwen2_5_vl_7b", tasks=task1))
print(metrics_for_model("qwen2_5_vl_7b", tasks=task2))

{'accuracy': 0.325, 'apd': 29.5, 'mtsd': 28.0}
{'accuracy': 0.041666666666666664, 'apd': 55.5, 'mtsd': 55.0}


In [4]:
df = fm_scores_for_model(MODEL, tasks=task1)

def pivot_fm_scores(df):
    df_long = df.melt(
        id_vars=['patient', 'fm_item'],
        value_vars=['pred_score', 'gt_score'],
        var_name='score_type',
        value_name='score'
    )
    df_long['col'] = df_long['score_type'] + "_" + df_long['patient']
    out = df_long.pivot(index="fm_item", columns="col", values="score")
    return out

pivot_fm_scores(df)

col,gt_score_C00011,gt_score_S0001,gt_score_S00021,gt_score_S0005,pred_score_C00011,pred_score_S0001,pred_score_S00021,pred_score_S0005
fm_item,,,,,,,,
3,2,2,1,2,1,1,0,0
4,2,2,0,2,1,2,1,1
5,2,2,1,2,2,1,1,1
6,2,2,0,2,2,2,2,2
7,2,2,0,2,2,2,2,2
8,2,1,0,2,2,2,1,1
9,2,2,0,2,1,1,1,1
10,2,1,0,2,1,1,1,1
11,2,2,0,2,1,1,1,1


In [5]:
df = answers_for_model(MODEL, tasks=task1)

In [6]:
def pivot_answers(df):
    melted_df = df.melt(
        id_vars=['patient', 'qid'],
        value_vars=['answer'],
        var_name='answer_type',
    )
    melted_df.drop(columns=['answer_type'], inplace=True)
    melted_df.rename({'value': 'answer'}, axis=1, inplace=True)
    out = (
        melted_df
        .pivot(index="qid", columns="patient", values="answer")
        .add_suffix("_answer")
    )
    return out

# df_long['col'] = df_long['score_type'] + "_" + df_long['patient']
# out = df_long.pivot(index="fm_item", columns="col", values="score")

In [48]:
pivot_answers(df).to_csv("scrap.csv", index=True)

In [7]:
"""
postprocess.ia.eval
===================

Evaluation pipeline utilities for StrokeRehab IA logs.

This module exposes three utilities for extracting information from a log:
answers to questions, FM scores computed by the answers programatically, and
model evaluation scores.
  • answers_for_model(model, tasks=…, logs_root=…, log_paths=None, drop_parsed=True)
  • fm_scores_for_model(model, tasks=…, logs_root=…, log_paths=None, …)
  • metrics_for_model(model, tasks=…, logs_root=…, log_paths=None, …)
      → Calls _aggregate_fm_metrics under the hood

Multi-model convenience
-----------------------
  • metrics_for_models(models="all", tasks=…, logs_root=…, …)
      → Tidy DataFrame with metrics per model

Export
------
  • metrics_df_to_latex(df, caption=None, label=None, float_format="%.2f")
      → Convert metrics DataFrame to LaTeX table

Notes
-----
• Input JSONL logs are expected to contain:
    {
      "doc": {"patient": "S0001"},
      "qids": "0<SEP>1<SEP>2<SEP>…",
      "filtered_resps": "<RESP><TIME> 0.00-2.33 ... <SEP> ..."
    }
• QIDs < 95 → raw string answer
• QIDs 95–96 → rounded average of numeric answers
• QIDs 97–100 → elapsed time when cumulative count hits threshold
"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterable, List, Optional, Set, Tuple, Union

import numpy as np
import pandas as pd
from data.utils_strokerehab import DataPaths

from typing import Sequence

# ─────────────────────────── Constants & Regex ─────────────────────────── #

SEP_RE = re.compile(r"\s*<SEP>\s*")
RESP_RE = re.compile(r"\s*<RESP>\s*")
TIME_RE = re.compile(r"<TIME>\s*([\d.]+)-([\d.]+)", re.IGNORECASE)

# Thresholds for the “timing” questions (used by FM item 33 logic downstream).
TIMING_THRESHOLDS = {97: 4, 98: 5, 99: 4, 100: 5}

# QIDs that compute elapsed times (their answers are floats / np.inf).
TIMING_QIDS = set(TIMING_THRESHOLDS.keys())


# ─────────────────────────── Parsing helpers ─────────────────────────── #

def _parse_filtered_resps(
    raw: Union[str, List[str]]
) -> List[List[Tuple[str, float, float]]]:
    """
    Split `filtered_resps` into blocks, preserving (answer, t_start, t_end).

    Parameters
    ----------
    raw
        Either a single string or a list of strings that should be concatenated.

    Returns
    -------
    blocks : list[list[(answer, t_start, t_end)]]
        A list of blocks; each block corresponds to one <RESP> … segment and
        contains answers aligned to the QID order for that record.
    """
    if isinstance(raw, list):
        raw = "".join(raw)

    blocks: List[List[Tuple[str, float, float]]] = []
    for chunk in RESP_RE.split(raw):
        chunk = chunk.strip()
        if not chunk:
            continue

        m = TIME_RE.search(chunk)
        if not m:
            raise ValueError("Missing <TIME start-end> tag in a <RESP> block.")
        t_start, t_end = map(float, m.groups())

        # strip time portion and split answers by <SEP>
        answers_part = TIME_RE.sub("", chunk).strip()
        answers = [a.strip() for a in SEP_RE.split(answers_part) if a.strip()]
        blocks.append([(ans, t_start, t_end) for ans in answers])

    return blocks


def _calc_answer(qid: int, triples: List[Tuple[str, float, float]]) -> object:
    """
    Convert a list of (answer, t_start, t_end) for a single QID into a final answer.

    Returns
    -------
    object
        • For qid < 95: the raw (first) answer string (or pandas.NA).
        • For qid in {95,96}: the rounded average (as string) of numeric answers (or pandas.NA).
        • For qid in {97..100}: t_end (float) when cumulative count reaches threshold, else np.inf.
        • Otherwise: pandas.NA.
    """
    # 0–94 → echo first raw answer
    if qid < 95:
        return triples[0][0] if triples else pd.NA

    # 95,96 → average of numeric answers (rounded to nearest int)
    if qid in (95, 96):
        nums = []
        for ans, _, _ in triples:
            try:
                nums.append(float(ans))
            except (TypeError, ValueError):
                pass
        return str(int(round(np.mean(nums)))) if nums else pd.NA

    # 97–100 → time when cumulative count hits threshold
    if qid in TIMING_QIDS:
        threshold = TIMING_THRESHOLDS[qid]
        cum = 0
        for ans, _t0, t_end in triples:
            try:
                cum += int(float(ans))
            except (TypeError, ValueError):
                continue
            if cum >= threshold:
                return float(t_end)
        return np.inf

    # Fallback
    return pd.NA


# ───────────────────────────── Public API ───────────────────────────── #

def _extract_answers(
    output_log_path: Union[str, Path, Iterable[Union[str, Path]]],
    drop_parsed: bool = True,
) -> pd.DataFrame:
    """
    Parse one or more JSONL logs and produce a table of (patient, qid, answer).

    Parameters
    ----------
    output_log_path
        Path or iterable of paths to JSONL files. Each line must contain:
        - rec["doc"]["patient"] : str
        - rec["qids"]           : "q0<SEP>q1<SEP>..."
        - rec["filtered_resps"] : string with <RESP> blocks and <TIME> tags
    drop_parsed
        If True, omit the 'parsed_response' column (human-readable).

    Returns
    -------
    DataFrame
        Columns: patient | qid | answer
        (If drop_parsed=False, also includes parsed_response: a compact string
         encoding the per-block answers and start/end times.)
    """
    paths: List[Path]
    if isinstance(output_log_path, (str, Path)):
        paths = [Path(output_log_path)]
    else:
        paths = [Path(p) for p in output_log_path]

    # Pass 1 – collect all patients and qids seen
    patients: Set[str] = set()
    qids_seen: Set[int] = set()
    for p in paths:
        with p.open(encoding="utf-8") as fh:
            for line in fh:
                rec = json.loads(line)
                patients.add(rec["doc"]["patient"])
                qids_seen.update(int(x) for x in SEP_RE.split(rec["qids"]))

    full_idx = pd.MultiIndex.from_product(
        [sorted(patients), sorted(qids_seen)], names=["patient", "qid"]
    )
    df = pd.DataFrame(index=full_idx).reset_index()
    df["answer"] = pd.NA
    df["parsed_response"] = pd.NA

    filled_pairs: Set[Tuple[str, int]] = set()

    # Pass 2 – compute parsed_response + final answer
    for p in paths:
        with p.open(encoding="utf-8") as fh:
            for line in fh:
                rec = json.loads(line)
                patient = rec["doc"]["patient"]
                qids_line = [int(x) for x in SEP_RE.split(rec["qids"])]

                blocks = _parse_filtered_resps(rec["filtered_resps"])
                if any(len(b) != len(qids_line) for b in blocks):
                    raise ValueError(
                        f"Answer/QID mismatch for patient={patient} (block length != qids length)."
                    )

                # transpose: per-qid triples across blocks
                per_qid: List[List[Tuple[str, float, float]]] = [[] for _ in qids_line]
                for blk in blocks:
                    for idx, triple in enumerate(blk):
                        per_qid[idx].append(triple)

                for idx, qid in enumerate(qids_line):
                    key = (patient, qid)
                    if key in filled_pairs:
                        raise ValueError(f"Duplicate entry for (patient={patient}, qid={qid}).")
                    triples = per_qid[idx]

                    parsed = " | ".join(f"{ans} {ts:.3f}-{te:.3f}" for ans, ts, te in triples)
                    ans = _calc_answer(qid, triples)

                    mask = (df["patient"] == patient) & (df["qid"] == qid)
                    df.loc[mask, ["parsed_response", "answer"]] = [parsed, ans]
                    filled_pairs.add(key)

    if drop_parsed:
        df.drop(columns=["parsed_response"], inplace=True)

    # Keep stable dtypes
    df["patient"] = df["patient"].astype(str)
    df["qid"] = df["qid"].astype(int)
    return df

In [32]:
output_log_path = "logs/strokerehab_ia3_3_30/qwen2_5_vl_7b/Qwen__Qwen2.5-VL-7B-Instruct/20250829_043621_samples_strokerehab_ia3_3_30.jsonl"
output_log_path = "/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab/logs/strokerehab_ia3_3_30/bot_8f/20250829_051922_samples_strokerehab_ia3_3_30.jsonl"
paths = [Path(output_log_path)]

# Pass 1 – collect all patients and qids seen
patients: Set[str] = set()
qids_seen: Set[int] = set()
for p in paths:
    with p.open(encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            patients.add(rec["doc"]["patient"])
            qids_seen.update(int(x) for x in SEP_RE.split(rec["qids"]))


In [33]:

full_idx = pd.MultiIndex.from_product(
    [sorted(patients), sorted(qids_seen)], names=["patient", "qid"]
)
df = pd.DataFrame(index=full_idx).reset_index()
df["answer"] = pd.NA
df["parsed_response"] = pd.NA

filled_pairs: Set[Tuple[str, int]] = set()


In [34]:

# Pass 2 – compute parsed_response + final answer
for p in paths:
    with p.open(encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            patient = rec["doc"]["patient"]
            qids_line = [int(x) for x in SEP_RE.split(rec["qids"])]

            blocks = _parse_filtered_resps(rec["filtered_resps"])
            if any(len(b) != len(qids_line) for b in blocks):
                raise ValueError(
                    f"Answer/QID mismatch for patient={patient} (block length != qids length)."
                )

            # transpose: per-qid triples across blocks
            per_qid: List[List[Tuple[str, float, float]]] = [[] for _ in qids_line]
            for blk in blocks:
                for idx, triple in enumerate(blk):
                    per_qid[idx].append(triple)

            for idx, qid in enumerate(qids_line):
                key = (patient, qid)
                if key in filled_pairs:
                    raise ValueError(f"Duplicate entry for (patient={patient}, qid={qid}).")
                triples = per_qid[idx]

                parsed = " | ".join(f"{ans} {ts:.3f}-{te:.3f}" for ans, ts, te in triples)
                ans = _calc_answer(qid, triples)

                mask = (df["patient"] == patient) & (df["qid"] == qid)
                df.loc[mask, ["parsed_response", "answer"]] = [parsed, ans]
                filled_pairs.add(key)

drop_parsed = False
if drop_parsed:
    df.drop(columns=["parsed_response"], inplace=True)

# Keep stable dtypes
df["patient"] = df["patient"].astype(str)
df["qid"] = df["qid"].astype(int)

In [20]:
len(qids_line)

26

In [21]:
len(blocks[0])

51

In [19]:
qids_line

[52,
 10,
 11,
 20,
 21,
 22,
 23,
 24,
 30,
 31,
 32,
 33,
 40,
 41,
 42,
 43,
 44,
 45,
 50,
 51,
 60,
 61,
 62,
 63,
 70,
 71]

In [18]:
blocks

[[('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('2', 0.0, 1.767),
  ('2', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('Yes', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('No', 0.0, 1.767),
  ('Yes', 0.0, 1

In [8]:
output_log_path = "logs/strokerehab_ia3_3_30/qwen2_5_vl_7b/Qwen__Qwen2.5-VL-7B-Instruct/20250829_043621_samples_strokerehab_ia3_3_30.jsonl"

_extract_answers(output_log_path)

ValueError: Answer/QID mismatch for patient=C00011 (block length != qids length).

In [4]:
df = fm_scores_for_model(MODEL, tasks=("strokerehab_ia3_3_30",))
pivot_fm_scores(df)

ValueError: Answer/QID mismatch for patient=C00011 (block length != qids length).

In [6]:
df = answers_for_model(model=MODEL, tasks=("strokerehab_ia3_3_30",))
patient_order = ['C00011', 'S0005', 'S0001', 'S00021']
patient_order = [p + '_answer' for p in patient_order]
pivot_answers(df)[patient_order]

ValueError: Answer/QID mismatch for patient=C00011 (block length != qids length).

In [32]:
import pandas as pd
from data.utils_strokerehab import DataPaths
pd.set_option("display.max_colwidth", None)
df = pd.read_csv("./data/ia/fm_item_questions3.csv")
df[df['fm_video'].str.contains("13")][['qid','question']]

,qid,question
0,10,Is the arm down at the side at onset? Answer 'Yes' or 'No' directly.
1,11,"At the beginning, is the arm down at the side? Answer 'Yes' or 'No' directly."
2,20,Is the elbow fully extended (straight) at onset? Answer 'Yes' or 'No' directly.
3,21,Is the elbow straight at onset? Answer 'Yes' or 'No' directly.
4,22,"At the beginning, is the elbow straight? Answer 'Yes' or 'No' directly."
5,23,"At the beginning, is the elbow mostly straight? Answer 'Yes' or 'No' directly."
6,24,Is the elbow mostly straight? Answer 'Yes' or 'No' directly.
7,30,Is the forearm in a neutral position (thumb pointing forward) at onset? Answer 'Yes' or 'No' directly.
8,31,"At the beginning, is the forearm in a neutral position (thumb pointing forward)? Answer 'Yes' or 'No' directly."
9,32,"At the beginning, is the forearm in a neutral position (thumb pointing in the same direction that the person is facing)? Answer 'Yes' or 'No' directly."
